# GPT Demo

Short notebook demonstrating: parameter count (with weight-tying savings), shape verification, text generation, and why untrained output is gibberish.

In [10]:
import torch
import tiktoken
from gpt_model import GPTModel, generate_text_simple_cached

GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
    "kv_window_size": 1024,
}

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval()

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): ModuleList(
    (0-11): 12 x TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
  )
  (final_norm): LayerNorm()
  (out_head): Linear(in_features=768, out_features=50257

## 1. Parameter count and weight-tying savings

In [11]:
total_params = sum(p.numel() for p in model.parameters())
vocab_size = GPT_CONFIG_124M["vocab_size"]
emb_dim = GPT_CONFIG_124M["emb_dim"]
tying_savings = vocab_size * emb_dim  # one shared matrix instead of tok_emb + out_head
tied_params = total_params - tying_savings

print(f"Total parameters (no tying): {total_params:,} (~{total_params/1e6:.1f}M)")
print(f"Weight-tying savings (vocab_size × emb_dim): {tying_savings:,} (~{tying_savings/1e6:.1f}M)")
print(f"With weight tying: {tied_params:,} (~{tied_params/1e6:.1f}M)")
print(f"  → 163M → 124M")

Total parameters (no tying): 163,009,536 (~163.0M)
Weight-tying savings (vocab_size × emb_dim): 38,597,376 (~38.6M)
With weight tying: 124,412,160 (~124.4M)
  → 163M → 124M


## 2. Shape verification

Feed input of shape `[2, 10]`; output logits should be `[2, 10, 50257]`.

In [12]:
batch_size, seq_len = 2, 10
dummy_idx = torch.randint(0, vocab_size, (batch_size, seq_len))
with torch.no_grad():
    logits = model(dummy_idx)

print(f"Input shape:  {dummy_idx.shape}  →  Output shape: {logits.shape}")
assert logits.shape == (batch_size, seq_len, vocab_size), f"Expected (2, 10, {vocab_size})"
print("✓ Shape [2, 10, 50257] verified.")

Input shape:  torch.Size([2, 10])  →  Output shape: torch.Size([2, 10, 50257])
✓ Shape [2, 10, 50257] verified.


## 3. Text generation

Generate from a prompt using `generate_text_simple` and decode the result.

In [14]:
tokenizer = tiktoken.get_encoding("gpt2")
prompt = "The meaning of life is"
encoded = tokenizer.encode(prompt)
idx = torch.tensor(encoded).unsqueeze(0)  # (1, T)

token_ids = generate_text_simple_cached(
    model=model,
    idx=idx,
    max_new_tokens=25,
)

decoded = tokenizer.decode(token_ids.squeeze(0).tolist())
print("Prompt:", prompt)
print("Generated (" + str(token_ids.shape[1]) + " tokens):", decoded)

Prompt: The meaning of life is
Generated (30 tokens): The meaning of life is Umb Dice architectLed Takeru Uriel fund Westbrookjen Δ exemplmissionCameraolkien figuring rigorous jerseys transpired Starr consumeneg maternity redevelopment transactions militants


## 4. Why is the output gibberish? What would make it coherent?

The model is **randomly initialized** and has never been trained: its weights do not encode language structure or facts, so next-token logits are effectively random and decoding produces nonsense. To get coherent text you need to **train the model** (e.g. with next-token prediction on a large text corpus) or **load pretrained weights** so that the parameters reflect real language statistics.